# Przygotowanie splitu

In [11]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

SEED = 42

[nltk_data] Downloading package stopwords to /home/janek/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [12]:
df_all = pd.read_csv("../data/raw/train.csv").dropna().drop_duplicates()

# derive levels of toxicity based on different columns
mild_toxic = df_all["toxic"].astype("bool") | df_all["obscene"].astype("bool") | df_all["insult"].astype("bool")
very_toxic = df_all["severe_toxic"].astype("bool") | df_all["threat"].astype("bool") | df_all["identity_hate"].astype("bool")

df_all["toxic_level"] = pd.Series([0 for _ in range(df_all.shape[0])])
df_all.loc[mild_toxic, "toxic_level"] = 1
df_all.loc[very_toxic, "toxic_level"] = 2

df_all.drop(["id", "toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"], axis=1, inplace=True)

In [13]:
sw = stopwords.words("english")

comments = df_all["comment_text"] \
    .apply(lambda x: re.sub(r'https?:\/\/.\S+', "", x)) \
    .apply(lambda x: x.lower()) \
    .apply(lambda x: re.sub(r"'", "", x)) \
    .apply(lambda x: re.sub(r"[^a-z]+", " ", x)) \
    .apply(lambda x: x.strip()) \
    .apply(lambda x: " ".join([y for y in x.split(" ") if y not in sw][:1024]))

df_all["comment_text"] = comments

**Opcja 1.** Po wydzieleniu próbek testowej i walidacyjnej wybieramy wszystkie możliwe bardzo toksyczne komentarze do treningu, a następnie dobieramy po równo komentarzy toksycznych i nietoksycznych do 20 000.

In [14]:
# validation and test data is sampled by random
df_val_test = df_all.sample(7000)
df_all.drop(df_val_test.index, inplace=True)

# we make sure to get as many comments with toxic_level=2 into the training set
df_train_severe = df_all[df_all["toxic_level"] == 2]
df_all.drop(df_train_severe.index, inplace=True)

df_all_toxic = df_all[df_all["toxic_level"] == 1]
df_all_nontoxic = df_all[df_all["toxic_level"] == 0]

df_train_toxic = df_all_toxic.sample((20000 - len(df_train_severe.index))//2)
df_train_nontoxic = df_all_nontoxic.sample(20000 - len(df_train_severe.index) - len(df_train_toxic.index))

df_train = pd.concat((df_train_severe, df_train_toxic, df_train_nontoxic)).sample(frac=1)

# save dataframes as files for future use
with open("../data/df_train.csv", "w") as f:
    df_train.to_csv(f, index=False)
with open("../data/df_val.csv", "w") as f:
    df_val_test.iloc[:2000].to_csv(f, index=False)
with open("../data/df_test.csv", "w") as f:
    df_val_test.iloc[2000:].to_csv(f, index=False)

**Opcja 2.** Po wydzieleniu próbek testowej i walidacyjnej wybieramy wszystkie możliwe bardzo toksyczne oraz toksyczne komentarze do treningu, a następnie dopełniamy nietoksycznymi do 20 000.

In [16]:
df_train_nontoxic_2 = df_all_nontoxic.sample(20000 - len(df_train_severe.index) - len(df_all_toxic.index))

df_train_2 = pd.concat([df_train_severe, df_train_nontoxic_2, df_all_toxic]).sample(frac=1)

with open("../data/df_train_2.csv", "w") as f:
    df_train.to_csv(f, index=False)